### Middleware
- Middleware provides aw ay to more tightly control what happens inside the agent.Middleware is useful for tracking the following:
  - Tracking agent behaviour with logging, analytics and debugging.
  - Transforming prompts, tool selection and output formatting
  - Adding retries, fallbacks and early termination logic
  - Applying rate limits, guardrails and PII detection.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")


### Summarization Middleware
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context.Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history
- Applications where preserving full conversation context matters

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

### Messagebased summarization
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model = "gpt-4o-mini",
        trigger=("messages",10),
        keep=("messages",4)
    )]
)


In [7]:
### Run with thread id
config={"configurable":{"thread_id":"test_1"}}


In [8]:
question = [
    "What is 1+1",
    "What is 5*10",
    "What is 12/4",
    "What is 7-3",
    "What is 9+6",
    "What is 8*7",
    "What is 15/3",
]

for q in question:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Message: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 1+1', additional_kwargs={}, response_metadata={}, id='b04362fd-d6e6-4843-bfb7-295e17092d05'), AIMessage(content='1 + 1 equals 2.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 13, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_060bd38f02', 'id': 'chatcmpl-EcEwfFJfdoX7cqoF5hrKgl4rR5HfC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a16a2b-c8a1-7592-b527-1a058e24d595-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 8, 'total_

In [9]:
# token size summarization
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,SystemMessage

@tool
def search_hotels(city: str)->str:
    """Search hotels - returns long responce to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotels - Excellent service and amenities.
    2. City Inn - Comfortable and affordable.
    3. Beachside Resort - Beautiful views and relaxing atmosphere.
    4. Mountain Lodge - Cozy and scenic.
    5. Downtown Suites - Convenient location and modern facilities."""

agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model = "gpt-4o-mini",
        trigger=("tokens",50),
        keep=("tokens",200)
    )]
)
config={"configurable":{"thread_id":"test_1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # Approximate token count assuming 4 characters per token


In [10]:
cities = ["New York", "Los Angeles", "Chicago", "Miami", "San Francisco"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Search hotels in {city}")]},config=config
    )
    print(f"Tokens used for {city}: {count_tokens(response['messages'])}")
    print(f"Response for {city}: {response['messages']}")

Tokens used for New York: 188
Response for New York: [HumanMessage(content='Search hotels in New York', additional_kwargs={}, response_metadata={}, id='028c3ff8-3dbc-4df7-ad89-f137b6e9079e'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 55, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_bdfeb08e5b', 'id': 'chatcmpl-EcFOWqf1ZnJbYh3bba8crWPumTlPx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a16a46-2a57-77e0-9b94-74ec29de9c87-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'New York'

### Human In the Loop Middleware
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human in the loop is useful for the following:
 - High-stakes operations requiring human approval(eg database writes, financial transactions).
 - Compliance workflows where human oversight is mandatory
 - Long-running conversations where human feedback guides the agent.

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Read email content by email ID"""
    return f"Content of email {email_id}"

def send_email_tool(email_id: str, content: str) -> str:
    """Send email content by email ID"""
    return f"Sent email {email_id} with content: {content}"



In [14]:
agent = create_agent(
    model="gpt-4o-mini",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email_tool":{ "allowed_decisions":["approve","edit","reject"] },
            "read_email_tool": False
        }
        
    )]
)

In [15]:
config = {"configurable":{"thread_id":"test-approve"}}
#step 1: Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@example.com with subject Meeting and content Let's meet tomorrow")]},config=config
)



In [16]:
result

{'messages': [HumanMessage(content="Send email to john@example.com with subject Meeting and content Let's meet tomorrow", additional_kwargs={}, response_metadata={}, id='906a0d98-f040-487b-a12b-f8c9122b05c0'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 87, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8a3291817b', 'id': 'chatcmpl-EcGNcPyNdEjpTrpMAXhurX4Z17Q2n', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a16a7f-fa39-7558-8ca4-485b7f63c049-0', tool_calls=[{'name': 'send_email_tool', 'arg

In [19]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused Approving....")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "approve"}
                ]

            }
        ),
        config=config
    )  
    print(f"Result:{result['messages'][-1].content}") 

Paused Approving....
Result:The email has been sent to john@example.com with the subject "Meeting" and the content "Let's meet tomorrow."


In [20]:
config = {"configurable":{"thread_id":"test-reject"}}
#step 1: Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@example.com with subject Meeting and content Let's meet tomorrow")]},config=config
)

In [21]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused Approving....")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type": "reject"}
                ]

            }
        ),
        config=config
    )  
    print(f"Result:{result['messages'][-1].content}") 

Paused Approving....
Result:It seems there was an issue with sending the email. Would you like me to try again or assist you with something else?
